In [1]:
# =========================
# LIBRARIES
# =========================

import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer
from pymongo import MongoClient
import os

In [6]:
# ==================================================
# LOAD DATA FROM MONGODB (DOCKER)
# ==================================================

client = MongoClient(
    host="localhost",
    port=27017,
    username="admin",
    password="oracle",
    authSource="admin"
)

db = client["redbull_racing"]
collection = db["merged_races"]

df = pd.DataFrame(list(collection.find()))

# Eliminar _id de MongoDB
if '_id' in df.columns:
    df = df.drop(columns=['_id'])

print(f"✅ Datos cargados desde MongoDB: {len(df)} registros")

✅ Datos cargados desde MongoDB: 0 registros


In [3]:
# ordenar cronológicamente

df = df.sort_values(by=["YEAR", "RACEID"])

KeyError: 'YEAR'

In [ ]:
## Feature Engineering

In [ ]:
df["ROLLING_POINTS"] = (
    df.groupby("DRIVERID")["POINTS"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

In [ ]:
df["ROLLING_LAP"] = (
    df.groupby("DRIVERID")["LAPMEAN"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

In [ ]:
df["LAP_CONSISTENCY"] = (
    df.groupby("DRIVERID")["LAPMEAN"]
    .transform(lambda x: x.rolling(5).std())
)

In [ ]:
df["RACE_INTERRUPTIONS"] = (
    df["SC_COUNT"] + df["PS_COUNT"]
)

In [ ]:
df["OVERTAKE_RATIO"] = (
    df["OVERTAKEN_POSITIONS_TOTAL"] / df["LAPS"]
)

In [ ]:
## EMBEDDINGS

In [ ]:
driver_encoder = LabelEncoder()
race_encoder = LabelEncoder()

df["DRIVER_ENCODED"] = driver_encoder.fit_transform(df["DRIVERREF"])
df["RACE_ENCODED"] = race_encoder.fit_transform(df["NAME_YEAR"])

In [ ]:
df.head()

,RACEID,DRIVER_POINTS_BEFORE_RACE,POINTS,DRIVERID,DRIVERREF,LAPS,MILLISECONDS,WEATHER_rain,WEATHER_WET,SCORE,...,PS_COUNT,SC_COUNT,YEAR,ROLLING_POINTS,ROLLING_LAP,LAP_CONSISTENCY,RACE_INTERRUPTIONS,OVERTAKE_RATIO,DRIVER_ENCODED,RACE_ENCODED
60762,833,NaN,-0.375795,579,fangio,62.0,7.105023e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,241,1
61007,833,NaN,-0.375795,589,chiron,26.0,2.979526e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,159,1
61606,833,NaN,-0.375795,619,gerard,67.0,7.678009e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,295,1
61776,833,NaN,-0.316728,627,rosier,68.0,7.792606e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.316728,NaN,NaN,NaN,0.0,678,1
62016,833,NaN,-0.375795,640,graffenried,36.0,4.125497e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,314,1


In [ ]:
df["ROLLING_POINTS"] = (
    df.groupby("DRIVERID")["POINTS"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
)

In [ ]:
# =========================
# EXPORT PROCESSED DATASET
# =========================

df.to_csv("processed_dataset.csv", index=False)

print("processed_dataset.csv saved successfully")

processed_dataset.csv saved successfully
